# 01 — Dataset Audit

**SIH PS 26168 — Intelligent Dead Reckoning**

> **Section 6 rule:** Before training, inspect the actual datasets. Do not guess schemas.
>
> **Section 7 rule:** Never replace real datasets with synthetic trajectories.
>
> **Section 5 rule:** Each dataset is audited independently and kept traceable to its original session.

**Outputs:**
- `artifacts/dataset_manifest.json` — machine-readable manifest of all five datasets
- `docs/dataset_audit.md` — human-readable audit report

---

## Sections
1. Environment & Configuration
2. IO-VNBD Audit (Primary)
3. GNSS Interference Dataset Audit
4. NavICGNSS Android Raw Measurements Audit
5. MOTOR Dataset Audit
6. OSM Map Data Audit
7. Cross-Dataset Summary & Train/Val/Test Split Strategy
8. Manifest Export

## 1. Environment & Configuration

In [ ]:
import os, sys, json, datetime, glob, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

# ── Paths from config ─────────────────────────────────────────────────────────
with open('configs/paths.yaml') as f:
    PATHS = yaml.safe_load(f)

PROJECT_ROOT = Path(os.getcwd())
ARTIFACTS    = PROJECT_ROOT / PATHS['artifacts_dir']
DOCS         = PROJECT_ROOT / PATHS['docs_dir']
PLOTS        = PROJECT_ROOT / PATHS['plots_dir']
ARTIFACTS.mkdir(exist_ok=True)
DOCS.mkdir(exist_ok=True)
(PLOTS / 'audit').mkdir(parents=True, exist_ok=True)

NOW = datetime.datetime.utcnow().isoformat()
print(f'Audit started : {NOW}')
print(f'Project root  : {PROJECT_ROOT}')
print(f'Artifacts dir : {ARTIFACTS}')

In [ ]:
# ── Audit helper utilities ─────────────────────────────────────────────────────

def folder_size_mb(path):
    """Recursively compute folder size in MB."""
    total = sum(f.stat().st_size for f in Path(path).rglob('*') if f.is_file())
    return round(total / 1e6, 2)

def list_files(path, exts=None, max_depth=4):
    """List files under path up to max_depth, optionally filtered by extension."""
    root = Path(path)
    files = []
    for f in root.rglob('*'):
        if not f.is_file(): continue
        depth = len(f.relative_to(root).parts)
        if depth > max_depth: continue
        if exts and f.suffix.lower() not in exts: continue
        files.append(f)
    return sorted(files)

def audit_csv(path, label=''):
    """Load a CSV and return shape, dtypes, missing stats, and sample."""
    try:
        df = pd.read_csv(path, nrows=50000)  # sample for speed
        info = {
            'file': str(path),
            'label': label,
            'rows_sampled': len(df),
            'columns': list(df.columns),
            'dtypes': df.dtypes.astype(str).to_dict(),
            'missing_count': df.isnull().sum().to_dict(),
            'missing_pct': (df.isnull().mean() * 100).round(2).to_dict(),
            'numeric_stats': df.describe().round(6).to_dict(),
        }
        return df, info
    except Exception as e:
        return None, {'file': str(path), 'error': str(e)}

def detect_sampling_rate(timestamps, unit='s', name=''):
    """Estimate sampling rate from timestamp column."""
    try:
        ts = pd.to_numeric(timestamps, errors='coerce').dropna().sort_values()
        diffs = ts.diff().dropna()
        median_dt = diffs.median()
        if median_dt <= 0:
            return None, None
        hz = round(1.0 / median_dt, 2) if unit == 's' else round(1000.0 / median_dt, 2)
        return round(median_dt, 6), hz
    except Exception as e:
        return None, None

manifest = {
    'audit_timestamp_utc': NOW,
    'project': 'SIH PS 26168 — Intelligent Dead Reckoning',
    'datasets': {}
}

print('Helpers loaded.')

## 2. IO-VNBD Audit (Primary Dataset)

IO-VNBD: Inertial and Odometry Vehicle Navigation Benchmark Dataset.
~40h / 1,300 km vehicle + ~58h / 4,400 km smartphone.
This is the **primary SIH benchmark dataset**.

In [ ]:
IO_ROOT = PROJECT_ROOT / PATHS['datasets']['io_vnbd']['root']
IO_SYNC = PROJECT_ROOT / PATHS['datasets']['io_vnbd']['synchronised']
IO_UNSYNC = PROJECT_ROOT / PATHS['datasets']['io_vnbd']['unsynchronised']

print(f'IO-VNBD root : {IO_ROOT}')
print(f'Exists       : {IO_ROOT.exists()}')
print(f'Size         : {folder_size_mb(IO_ROOT)} MB')
print()

# List driver categories
if IO_SYNC.exists():
    drivers = sorted([d.name for d in IO_SYNC.iterdir() if d.is_dir()])
    print(f'Driver categories (Synchronised): {drivers}')
    for drv in drivers:
        sessions = sorted([s.name for s in (IO_SYNC / drv).iterdir() if s.is_dir()])
        print(f'  {drv}: {sessions}')

In [ ]:
# ── Audit each IO-VNBD CSV session ──────────────────────────────────────────
io_sessions = []
csv_files = list_files(IO_SYNC, exts=['.csv']) if IO_SYNC.exists() else []

print(f'Found {len(csv_files)} CSV files in synchronised IO-VNBD:')
for f in csv_files:
    print(f'  {f.relative_to(IO_ROOT)}')

print()

for csv_path in csv_files[:20]:  # audit up to 20 sessions
    df, info = audit_csv(csv_path, label=csv_path.stem)
    if df is None:
        print(f'ERROR reading {csv_path.name}: {info.get("error")}')
        continue

    cols = info['columns']
    print(f'\n--- {csv_path.name} ---')
    print(f'  Rows (sampled): {info["rows_sampled"]:,}')
    print(f'  Columns ({len(cols)}): {cols[:10]}{'...' if len(cols)>10 else ''}')

    # Try to detect timestamp column
    ts_candidates = [c for c in cols if 'time' in c.lower() or 'ts' in c.lower() or 't_' in c.lower()]
    print(f'  Timestamp candidates: {ts_candidates}')
    if ts_candidates:
        dt, hz = detect_sampling_rate(df[ts_candidates[0]])
        print(f'  Inferred sample interval: {dt}s → {hz} Hz')

    # IMU columns
    imu_cols = [c for c in cols if any(k in c.lower() for k in ['acc','gyro','mag','gyr','imu'])]
    gnss_cols = [c for c in cols if any(k in c.lower() for k in ['lat','lon','alt','gps','gnss','speed','hdop','pdop'])]
    gt_cols = [c for c in cols if any(k in c.lower() for k in ['gt','ground','truth','ref','vel','vx','vy'])]
    print(f'  IMU cols  : {imu_cols}')
    print(f'  GNSS cols : {gnss_cols}')
    print(f'  GT cols   : {gt_cols}')

    missing_pct = {k: v for k, v in info['missing_pct'].items() if v > 0}
    if missing_pct:
        print(f'  Missing % : {missing_pct}')

    io_sessions.append({
        'file': str(csv_path.relative_to(PROJECT_ROOT)),
        'stem': csv_path.stem,
        'rows_sampled': info['rows_sampled'],
        'columns': cols,
        'timestamp_candidates': ts_candidates,
        'imu_columns': imu_cols,
        'gnss_columns': gnss_cols,
        'ground_truth_columns': gt_cols,
        'missing_pct': missing_pct,
    })

print(f'\nAudited {len(io_sessions)} IO-VNBD sessions.')

In [ ]:
# ── Visualise a sample IO-VNBD session ───────────────────────────────────────
if csv_files:
    sample_csv = csv_files[0]
    df_s, _ = audit_csv(sample_csv)
    if df_s is not None:
        print(f'Visualising: {sample_csv.name}')
        print(df_s.head(3).to_string())
        print('\nData types:')
        print(df_s.dtypes.to_string())
        print('\nBasic stats:')
        print(df_s.describe().round(4).to_string())

In [ ]:
# ── Record IO-VNBD in manifest ─────────────────────────────────────────────
manifest['datasets']['io_vnbd'] = {
    'source_folder': str(IO_ROOT.relative_to(PROJECT_ROOT)),
    'role': 'Primary real vehicle dataset for official SIH inertial-odometry benchmarking',
    'primary': True,
    'readme': str((IO_ROOT / 'README.md').relative_to(PROJECT_ROOT)),
    'synchronised_root': str(IO_SYNC.relative_to(PROJECT_ROOT)) if IO_SYNC.exists() else None,
    'size_mb': folder_size_mb(IO_ROOT),
    'sessions': io_sessions,
    'notes': [
        'Schemas, sampling rates, and ground-truth quality must be confirmed from full file read.',
        'Session-level train/val/test splits must be defined after full audit.',
        'Ground truth must not be used as model input during GNSS-denied inference.',
    ]
}
print('IO-VNBD recorded in manifest.')

## 3. GNSS Interference Dataset Audit

In [ ]:
GNSS_ROOT = PROJECT_ROOT / PATHS['datasets']['gnss_interference']['root']
print(f'GNSS interference root : {GNSS_ROOT}')
print(f'Exists                 : {GNSS_ROOT.exists()}')

gnss_files = list_files(GNSS_ROOT) if GNSS_ROOT.exists() else []
print(f'Total files            : {len(gnss_files)}')
print(f'Size                   : {folder_size_mb(GNSS_ROOT) if GNSS_ROOT.exists() else "N/A"} MB')

# Show directory structure
if GNSS_ROOT.exists():
    for item in sorted(GNSS_ROOT.rglob('*')):
        depth = len(item.relative_to(GNSS_ROOT).parts)
        if depth <= 3:
            indent = '  ' * (depth - 1)
            tag = '/' if item.is_dir() else f' ({item.stat().st_size/1e3:.0f} KB)'
            print(f'{indent}{item.name}{tag}')

In [ ]:
gnss_sessions = []
gnss_csvs = list_files(GNSS_ROOT, exts=['.csv', '.txt', '.log']) if GNSS_ROOT.exists() else []

for f in gnss_csvs[:10]:
    df, info = audit_csv(f, label=f.stem)
    if df is not None:
        print(f'{f.name}: {info["rows_sampled"]} rows | cols: {info["columns"][:8]}')
        gnss_sessions.append({
            'file': str(f.relative_to(PROJECT_ROOT)),
            'rows_sampled': info['rows_sampled'],
            'columns': info['columns'],
        })

manifest['datasets']['gnss_interference'] = {
    'source_folder': str(GNSS_ROOT.relative_to(PROJECT_ROOT)) if GNSS_ROOT.exists() else PATHS['datasets']['gnss_interference']['root'],
    'role': 'GNSS interference, jamming, and spoofing experiments',
    'primary': False,
    'size_mb': folder_size_mb(GNSS_ROOT) if GNSS_ROOT.exists() else 0,
    'sessions': gnss_sessions,
    'notes': [
        'Use for GNSS quality/degradation detection experiments only.',
        'Schema must be validated before use in any training pipeline.',
    ]
}
print('GNSS interference recorded in manifest.')

## 4. NavICGNSS Android Raw Measurements Audit

In [ ]:
NAVIC_ROOT = PROJECT_ROOT / PATHS['datasets']['navic_gnss']['root']
print(f'NavICGNSS root : {NAVIC_ROOT}')
print(f'Exists         : {NAVIC_ROOT.exists()}')

navic_files = list_files(NAVIC_ROOT) if NAVIC_ROOT.exists() else []
print(f'Total files    : {len(navic_files)}')
print(f'Size           : {folder_size_mb(NAVIC_ROOT) if NAVIC_ROOT.exists() else "N/A"} MB')

if NAVIC_ROOT.exists():
    for item in sorted(NAVIC_ROOT.rglob('*')):
        depth = len(item.relative_to(NAVIC_ROOT).parts)
        if depth <= 4:
            indent = '  ' * (depth - 1)
            tag = '/' if item.is_dir() else f' ({item.stat().st_size/1e3:.0f} KB)'
            print(f'{indent}{item.name}{tag}')

In [ ]:
navic_sessions = []
navic_csvs = list_files(NAVIC_ROOT, exts=['.csv', '.txt', '.log', '.nmea']) if NAVIC_ROOT.exists() else []

for f in navic_csvs[:10]:
    df, info = audit_csv(f, label=f.stem)
    if df is not None:
        print(f'{f.name}: {info["rows_sampled"]} rows | cols: {info["columns"][:8]}')
        navic_sessions.append({
            'file': str(f.relative_to(PROJECT_ROOT)),
            'rows_sampled': info['rows_sampled'],
            'columns': info['columns'],
        })

# Check for ground truth reference
gt_file = NAVIC_ROOT.parent / 'ublox_ground_truth_positions.txt'
gt_info = None
if gt_file.exists():
    gt_df, gt_info_raw = audit_csv(gt_file, label='ground_truth')
    print(f'\nGround truth file: {gt_file.name}')
    if gt_df is not None:
        print(gt_df.head())
    gt_info = {'file': str(gt_file.relative_to(PROJECT_ROOT))}

manifest['datasets']['navic_gnss'] = {
    'source_folder': str(NAVIC_ROOT.relative_to(PROJECT_ROOT)) if NAVIC_ROOT.exists() else PATHS['datasets']['navic_gnss']['root'],
    'role': 'Android/NavIC raw GNSS measurement experiments and GNSS-side validation',
    'primary': False,
    'size_mb': folder_size_mb(NAVIC_ROOT) if NAVIC_ROOT.exists() else 0,
    'ground_truth_file': gt_info,
    'sessions': navic_sessions,
    'notes': [
        'Validate timestamp and coordinate system before use.',
        'Ground truth reference: ublox_ground_truth_positions.txt (needs schema audit).',
    ]
}
print('NavICGNSS recorded in manifest.')

## 5. MOTOR Dataset Audit

In [ ]:
MOTOR_ROOT  = PROJECT_ROOT / PATHS['datasets']['motor']['root']
MOTOR_ANN   = PROJECT_ROOT / PATHS['datasets']['motor']['annotations']
print(f'MOTOR root       : {MOTOR_ROOT}')
print(f'Exists           : {MOTOR_ROOT.exists()}')
print(f'Size             : {folder_size_mb(MOTOR_ROOT) if MOTOR_ROOT.exists() else "N/A"} MB')

if MOTOR_ROOT.exists():
    for item in sorted(MOTOR_ROOT.rglob('*')):
        depth = len(item.relative_to(MOTOR_ROOT).parts)
        if depth <= 3 and item.is_file():
            indent = '  ' * (depth - 1)
            print(f'{indent}{item.name} ({item.stat().st_size/1e3:.0f} KB)')
        elif depth <= 2 and item.is_dir():
            indent = '  ' * (depth - 1)
            print(f'{indent}{item.name}/')

In [ ]:
motor_ann_info = None
if MOTOR_ANN.exists():
    df_m, info_m = audit_csv(MOTOR_ANN, label='motor_annotations')
    if df_m is not None:
        print(f'annotations.csv: {info_m["rows_sampled"]} rows')
        print(f'Columns: {info_m["columns"]}')
        print(df_m.head(3).to_string())
        print('\nDtypes:')
        print(df_m.dtypes.to_string())
        motor_ann_info = {
            'rows_sampled': info_m['rows_sampled'],
            'columns': info_m['columns'],
            'missing_pct': info_m['missing_pct'],
        }

clips_dir = MOTOR_ROOT / 'clips'
n_clips = len(list(clips_dir.rglob('*'))) if clips_dir.exists() else 0

manifest['datasets']['motor'] = {
    'source_folder': str(MOTOR_ROOT.relative_to(PROJECT_ROOT)) if MOTOR_ROOT.exists() else PATHS['datasets']['motor']['root'],
    'role': 'Additional real vehicle/motorcycle data — schema must be audited before use',
    'primary': False,
    'size_mb': folder_size_mb(MOTOR_ROOT) if MOTOR_ROOT.exists() else 0,
    'annotations_info': motor_ann_info,
    'clips_file_count': n_clips,
    'notes': [
        'Only use after sensor schema, timestamps, coordinate system, and ground truth are audited.',
        'Clips directory: video or other media files — determine format before use.',
    ]
}
print('MOTOR recorded in manifest.')

## 6. OSM Map Data Audit

In [ ]:
OSM_ROOT = PROJECT_ROOT / PATHS['datasets']['osm']['root']
PBF_FILE = PROJECT_ROOT / PATHS['datasets']['osm']['pbf_file']

print(f'OSM root   : {OSM_ROOT}')
print(f'PBF file   : {PBF_FILE}')
print(f'PBF exists : {PBF_FILE.exists()}')

pbf_size_mb = PBF_FILE.stat().st_size / 1e6 if PBF_FILE.exists() else 0
print(f'PBF size   : {pbf_size_mb:.1f} MB')

# Check osmium / pyosmium is available for parsing
try:
    import osmium
    print(f'pyosmium   : {osmium.__version__} — available for PBF parsing')
except ImportError:
    print('pyosmium   : NOT installed — needed for PBF parsing (pip install osmium)')

try:
    import osmnx as ox
    print(f'osmnx      : {ox.__version__} — available')
except ImportError:
    print('osmnx      : NOT installed (pip install osmnx)')

manifest['datasets']['osm'] = {
    'source_folder': str(OSM_ROOT.relative_to(PROJECT_ROOT)) if OSM_ROOT.exists() else PATHS['datasets']['osm']['root'],
    'role': 'Offline OpenStreetMap road data for GNN map matching — not a trajectory dataset',
    'primary': False,
    'pbf_file': str(PBF_FILE.relative_to(PROJECT_ROOT)) if PBF_FILE.exists() else None,
    'pbf_size_mb': pbf_size_mb,
    'notes': [
        'india-260912.osm.pbf covers India road network.',
        'Local sub-graphs will be extracted per session bounding box during GNN training.',
        'Do not load entire India graph into memory for inference — use local radius queries.',
    ]
}
print('OSM recorded in manifest.')

## 7. Cross-Dataset Summary & Train/Val/Test Split Strategy

Per Section 9: use **session/trajectory-level** splits only. Never random sample-level splits.

In [ ]:
print('=' * 60)
print('DATASET SUMMARY')
print('=' * 60)
for ds_name, ds in manifest['datasets'].items():
    size = ds.get('size_mb', 0)
    sessions = ds.get('sessions', [])
    primary = '★ PRIMARY' if ds.get('primary') else ''
    print(f'\n{ds_name.upper()} {primary}')
    print(f'  Role    : {ds["role"]}')
    print(f'  Size    : {size:.1f} MB')
    print(f'  Sessions: {len(sessions)}')
    for note in ds.get('notes', []):
        print(f'  ⚠ {note}')

In [ ]:
# ── Proposed session-level split strategy ────────────────────────────────────
# NOTE: Final splits are determined AFTER full schema validation.
# This is a preliminary strategy only.

split_strategy = {
    'method': 'session_level',
    'rationale': 'Adjacent samples from the same trajectory must never be split randomly. '
                 'Session-level splitting prevents data leakage.',
    'io_vnbd': {
        'train_drivers': ['M (Driver B)', 'Vf (Driver E)', 'Vta (Driver E)', 'Vtb (Driver E)', 'Vw (Driver E)'],
        'val_drivers':   ['Y (Driver D)'],
        'test_drivers':  ['S (Driver A)'],
        'note': 'Preliminary assignment — subject to revision after full schema/ground-truth validation. '
                'Test set (Driver A / S sessions) must remain completely untouched until final evaluation.'
    },
    'final_test_set_locked': True,
    'warning': 'Do NOT use final test sessions for ANY hyperparameter tuning, '
               'architecture selection, or threshold selection.'
}

manifest['split_strategy'] = split_strategy

print('Proposed split strategy:')
print(json.dumps(split_strategy, indent=2))

## 8. Manifest Export

In [ ]:
# Save dataset_manifest.json
manifest_path = ARTIFACTS / 'dataset_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2, default=str)
print(f'Manifest saved: {manifest_path}')

In [ ]:
# Save human-readable dataset_audit.md
audit_md_path = DOCS / 'dataset_audit.md'

lines = [
    '# Dataset Audit Report',
    '',
    f'**SIH PS 26168 — Intelligent Dead Reckoning**',
    f'Generated: {NOW}',
    '',
    '---',
    '',
]

for ds_name, ds in manifest['datasets'].items():
    primary_tag = ' ★ PRIMARY' if ds.get('primary') else ''
    lines += [
        f'## {ds_name.upper()}{primary_tag}',
        '',
        f'**Role:** {ds["role"]}  ',
        f'**Source:** `{ds["source_folder"]}`  ',
        f'**Size:** {ds.get("size_mb", 0):.1f} MB  ',
        f'**Sessions found:** {len(ds.get("sessions", []))}  ',
        '',
    ]
    if ds.get('sessions'):
        lines.append('| File | Rows | Columns | Timestamp | IMU | GNSS | GT |')
        lines.append('|---|---|---|---|---|---|---|')
        for s in ds['sessions'][:20]:
            ncols = len(s.get('columns', []))
            ts = ', '.join(s.get('timestamp_candidates', []))[:30]
            imu = len(s.get('imu_columns', []))
            gnss = len(s.get('gnss_columns', []))
            gt = len(s.get('ground_truth_columns', []))
            rows = s.get('rows_sampled', '?')
            fname = Path(s['file']).name
            lines.append(f'| `{fname}` | {rows:,} | {ncols} | {ts} | {imu} | {gnss} | {gt} |')
        lines.append('')
    for note in ds.get('notes', []):
        lines.append(f'> ⚠ {note}  ')
    lines += ['', '---', '']

lines += [
    '## Train / Validation / Test Split Strategy',
    '',
    f'**Method:** {manifest["split_strategy"]["method"]}  ',
    f'**Rationale:** {manifest["split_strategy"]["rationale"]}  ',
    '',
    f'> **WARNING:** {manifest["split_strategy"]["warning"]}  ',
    '',
]

with open(audit_md_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

print(f'Audit report saved: {audit_md_path}')
print()
print('=' * 60)
print('  DATASET AUDIT COMPLETE')
print('  Next: review artifacts/dataset_manifest.json')
print('  Then: await approval before proceeding to Phase 1')
print('=' * 60)